# ResMill Interactive Explorer

Single interactive notebook for all ResMill layer types. Curated
parameter set per layer — only the knobs that visibly change the
reservoir, mirroring `examples/dataset_generation/config_*.json`.
Use this notebook to verify each parameter's effect and pick ranges
before running the dataset generator at scale.

## Layout

1. **Grid + seed** — shared by all layers.
2. **Layer type dropdown** — Lobe, Gaussian, Meandering channel,
   Braided channel, Delta.
3. **Parameters** — each layer's own curated panel. Channels have a
   **Preset / Custom** toggle: pick a preset to load the canonical
   Pyrcz-2004 baseline, then tweak any knob on top.
4. **Generate** — runs `create_geology` and renders 4 panels:
   plan-view sand thickness, XY mid-Z, XZ mid-Y, YZ mid-X.

Default mode is **manual**: drag sliders freely, click *Generate* to
render. Tick *Auto-regenerate* for live updates on every change.

In [ ]:
import time, copy
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import resmill as rm
from resmill.layers.channel import (
    PV_SHOESTRING, CB_JIGSAW, CB_LABYRINTH, SH_DISTAL, SH_PROXIMAL,
)
from resmill.layers.delta import DELTA_FAN

%matplotlib inline

MEANDER_PRESETS = {
    'PV_SHOESTRING': PV_SHOESTRING,
    'CB_LABYRINTH':  CB_LABYRINTH,
    'SH_DISTAL':     SH_DISTAL,
    'SH_PROXIMAL':   SH_PROXIMAL,
}
BRAIDED_PRESETS = {'CB_JIGSAW': CB_JIGSAW}
DELTA_PRESETS = {'DELTA_FAN': DELTA_FAN}

STYLE  = {'description_width': '170px'}
LAYOUT = widgets.Layout(width='480px')

In [ ]:
# ── Shared grid widgets (default 64×64×32, same as tutorials + dataset) ──
w_nx    = widgets.IntSlider(value=64,  min=32, max=128, step=8, description='nx', style=STYLE, layout=LAYOUT)
w_ny    = widgets.IntSlider(value=64,  min=32, max=128, step=8, description='ny', style=STYLE, layout=LAYOUT)
w_nz    = widgets.IntSlider(value=32,  min=16, max=64,  step=4, description='nz', style=STYLE, layout=LAYOUT)
w_xlen  = widgets.FloatSlider(value=640., min=320., max=4000., step=80., description='x_len (m)', style=STYLE, layout=LAYOUT)
w_ylen  = widgets.FloatSlider(value=640., min=320., max=4000., step=80., description='y_len (m)', style=STYLE, layout=LAYOUT)
w_zlen  = widgets.FloatSlider(value=32.,  min=10.,  max=80.,   step=2.,  description='z_len (m)', style=STYLE, layout=LAYOUT)
w_depth = widgets.FloatSlider(value=0.,   min=0.,   max=3000., step=100, description='top_depth (m)', style=STYLE, layout=LAYOUT)
w_dip   = widgets.FloatSlider(value=0.,   min=0.,   max=10.,   step=0.5, description='dip (deg)', style=STYLE, layout=LAYOUT)
w_seed  = widgets.IntText(value=42, description='seed', style=STYLE, layout=LAYOUT)

grid_box = widgets.HBox([
    widgets.VBox([w_nx, w_ny, w_nz, w_seed]),
    widgets.VBox([w_xlen, w_ylen, w_zlen, w_depth, w_dip]),
])

In [ ]:
# ── Per-layer curated parameter panels ────────────────────────────
# Same knobs as examples/dataset_generation/config_*.json so the
# values you pick here translate directly to dataset ranges.

def _slider(value, lo, hi, step, desc):
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step,
                                description=desc, style=STYLE, layout=LAYOUT,
                                continuous_update=False)
def _islider(value, lo, hi, step, desc):
    return widgets.IntSlider(value=value, min=lo, max=hi, step=step,
                              description=desc, style=STYLE, layout=LAYOUT,
                              continuous_update=False)

# ---- Lobe ------------------------------------------------------------------
lobe_w = dict(
    poro_ave    = _slider(0.25, 0.10, 0.40, 0.01, 'poro_ave'),
    perm_ave    = _slider(2.0,  0.0,  4.0,  0.1,  'perm_ave (log10 mD)'),
    poro_std    = _slider(0.03, 0.01, 0.05, 0.005, 'poro_std'),
    perm_std    = _slider(0.6,  0.1,  1.5,  0.05, 'perm_std (log10 mD)'),
    ntg         = _slider(0.5,  0.2,  0.8,  0.05, 'ntg'),
    dhmin       = _islider(3,   2,    5,    1,    'dhmin (m)'),
    dhmax       = _islider(8,   6,    10,   1,    'dhmax (m)'),
    rmin        = _slider(30.,  20.,  45.,  1.,   'rmin (m)'),
    rmax        = _slider(60.,  50.,  80.,  2.,   'rmax (m)'),
    asp         = _slider(1.5,  1.0,  2.5,  0.1,  'asp'),
    bouma_factor= _slider(1.0,  0.0,  2.0,  0.1,  'bouma_factor'),
    upthinning  = widgets.Checkbox(value=True, description='upthinning', style=STYLE, layout=LAYOUT),
)
lobe_box = widgets.VBox(list(lobe_w.values()))

# ---- Gaussian --------------------------------------------------------------
gauss_w = dict(
    poro_ave       = _slider(0.25, 0.10, 0.40, 0.01, 'poro_ave'),
    perm_ave       = _slider(2.0,  0.0,  4.0,  0.1,  'perm_ave (log10 mD)'),
    poro_std       = _slider(0.03, 0.01, 0.05, 0.005, 'poro_std'),
    perm_std       = _slider(0.6,  0.1,  1.5,  0.05, 'perm_std (log10 mD)'),
    ntg            = _slider(0.5,  0.2,  0.8,  0.05, 'ntg'),
    nugget         = _slider(0.05, 0.01, 0.15, 0.01, 'nugget'),
    poro_perm_corr = _slider(0.6,  0.3,  0.9,  0.05, 'poro_perm_corr'),
)
gauss_box = widgets.VBox(list(gauss_w.values()))

# ---- Meandering channel (preset + curated overrides) -----------------------
mean_preset = widgets.Dropdown(
    options=['Custom'] + list(MEANDER_PRESETS.keys()),
    value='PV_SHOESTRING', description='preset', style=STYLE, layout=LAYOUT)
mean_w = dict(
    mCHsinu        = _slider(1.6,  1.10, 1.90, 0.02, 'mCHsinu'),
    mCHwdratio     = _slider(10.,  6.,   24.,  0.5,  'mCHwdratio'),
    mCHdepth       = _slider(2.5,  1.0,  6.0,  0.1,  'mCHdepth (m)'),
    probAvulInside = _slider(0.05, 0.0,  0.6,  0.02, 'probAvulInside'),
    mFFCHprop      = _slider(0.0,  0.0,  0.7,  0.05, 'mFFCHprop'),
    mdistMigrate   = _slider(35.,  10.,  60.,  1.,   'mdistMigrate (m)'),
    NTGtarget      = _slider(0.10, 0.05, 0.80, 0.02, 'NTGtarget'),
    azimuth        = _slider(0.,   0.,   360., 5.,   'azimuth (deg)'),
)
mean_box = widgets.VBox([mean_preset] + list(mean_w.values()))

# ---- Braided channel (preset + curated overrides) --------------------------
braid_preset = widgets.Dropdown(
    options=['Custom'] + list(BRAIDED_PRESETS.keys()),
    value='CB_JIGSAW', description='preset', style=STYLE, layout=LAYOUT)
braid_w = dict(
    mCHsinu        = _slider(1.3,  1.10, 1.50, 0.02, 'mCHsinu'),
    mCHwdratio     = _slider(16.,  10.,  24.,  0.5,  'mCHwdratio'),
    mCHdepth       = _slider(3.0,  1.5,  5.0,  0.1,  'mCHdepth (m)'),
    probAvulInside = _slider(0.40, 0.05, 0.60, 0.02, 'probAvulInside'),
    mFFCHprop      = _slider(0.5,  0.0,  0.8,  0.05, 'mFFCHprop'),
    NTGtarget      = _slider(0.30, 0.10, 0.60, 0.02, 'NTGtarget'),
    azimuth        = _slider(0.,   0.,   360., 5.,   'azimuth (deg)'),
)
braid_box = widgets.VBox([braid_preset] + list(braid_w.values()))

# ---- Delta (fluvial-engine; baseline = DELTA_FAN) -------------------------
delta_w = dict(
    trunk_length_fraction = _slider(0.4, 0.0, 0.8, 0.05, 'trunk_length_fraction'),
    progradation_fraction = _slider(0.0, 0.0, 0.6, 0.05, 'progradation_fraction'),
    branch_spread_deg     = _slider(0.0, 0.0, 60., 2.5,  'branch_spread_deg'),
    n_generations         = _islider(8, 1, 16, 1,        'n_generations'),
    ntime_per_gen         = _islider(80, 20, 160, 10,    'ntime_per_gen'),
    mCHsinu               = _slider(1.10, 1.0, 1.40, 0.02, 'mCHsinu'),
    mFFCHprop             = _slider(0.0, 0.0, 0.5, 0.05,  'mFFCHprop'),
    paint_mouth_bars      = widgets.Checkbox(value=False,
                                              description='paint_mouth_bars',
                                              style=STYLE, layout=LAYOUT),
    azimuth               = _slider(0., 0., 360., 5.,     'azimuth (deg)'),
)
delta_box = widgets.VBox(list(delta_w.values()))

_PANELS = {
    'Lobe': (lobe_box, lobe_w, None, None),
    'Gaussian': (gauss_box, gauss_w, None, None),
    'Meandering': (mean_box, mean_w, mean_preset, MEANDER_PRESETS),
    'Braided': (braid_box, braid_w, braid_preset, BRAIDED_PRESETS),
    'Delta': (delta_box, delta_w, None, None),
}

def _apply_preset(name, widget_dict, preset_table):
    if name == 'Custom' or preset_table is None:
        return
    preset = preset_table[name]
    for key, w in widget_dict.items():
        if key in preset:
            try:
                w.value = preset[key]
            except Exception:
                pass

for label, (_, w_dict, preset_w, preset_table) in _PANELS.items():
    if preset_w is None:
        continue
    def _make_handler(table, wd):
        def _h(change):
            _apply_preset(change['new'], wd, table)
        return _h
    preset_w.observe(_make_handler(preset_table, w_dict), names='value')
    _apply_preset(preset_w.value, w_dict, preset_table)

In [ ]:
# ── Layer build + render ───────────────────────────────────────────

model_selector = widgets.Dropdown(
    options=list(_PANELS.keys()), value='Meandering',
    description='Layer type', style=STYLE, layout=LAYOUT)
param_area = widgets.Output()
output_area = widgets.Output(layout=widgets.Layout(
    min_height='560px', border='1px solid #d0d0d0', padding='4px'))
status_label = widgets.HTML(value='<i style="color:#666">Click Generate to render.</i>')
w_auto = widgets.Checkbox(value=False,
    description='Auto-regenerate on slider change',
    style={'description_width': 'initial'})
w_gen_btn = widgets.Button(description='Generate', button_style='primary',
                            icon='play', layout=widgets.Layout(width='160px'))

def _render_panel(*_):
    panel, *_unused = _PANELS[model_selector.value]
    with param_area:
        clear_output(wait=True)
        display(panel)
model_selector.observe(_render_panel, names='value')

def _kwargs_for(model_type):
    """Curated kwargs from the active panel; preset baseline + slider overrides."""
    _, w_dict, preset_w, preset_table = _PANELS[model_type]
    out = {k: w.value for k, w in w_dict.items()}
    if preset_w is not None and preset_w.value != 'Custom':
        baseline = dict(preset_table[preset_w.value])
        baseline.update(out)
        out = baseline
    return out

def _run():
    model = model_selector.value
    grid_kw = dict(nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
                    x_len=w_xlen.value, y_len=w_ylen.value,
                    z_len=w_zlen.value, top_depth=w_depth.value, dip=w_dip.value)
    seed = int(w_seed.value)
    np.random.seed(seed)
    if model == 'Lobe':
        layer = rm.LobeLayer(**grid_kw); layer.create_geology(**_kwargs_for(model))
    elif model == 'Gaussian':
        layer = rm.GaussianLayer(**grid_kw); layer.create_geology(**_kwargs_for(model))
    elif model == 'Meandering':
        layer = rm.MeanderingChannelLayer(**grid_kw); layer.create_geology(seed=seed, **_kwargs_for(model))
    elif model == 'Braided':
        layer = rm.BraidedChannelLayer(**grid_kw); layer.create_geology(seed=seed, **_kwargs_for(model))
    elif model == 'Delta':
        layer = rm.DeltaLayer(**grid_kw); layer.create_geology(seed=seed, **_kwargs_for(model))
    return layer

_busy = {'b': False}
def _generate(*_, force=False):
    if not force and not w_auto.value: return
    if _busy['b']: return
    _busy['b'] = True
    try:
        status_label.value = '<i style="color:#0a6">⏳ generating…</i>'
        t0 = time.perf_counter()
        try:
            layer = _run()
        except Exception as e:
            import traceback
            status_label.value = f'<span style="color:red">Error: {e}</span>'
            output_area.clear_output(wait=False)
            with output_area:
                print(traceback.format_exc())
            return
        plt.close('all')
        # Library viewer: 8 slices on each of X/Y/Z by default.
        with output_area:
            clear_output(wait=True)
            rm.plot_slices(layer, facies='binary',
                title=f'{model_selector.value}  —  NTG = {layer.active.mean():.2%}')
        dt = time.perf_counter() - t0
        status_label.value = f'<span style="color:#0a6">✓ {dt:.2f}s</span>'
    finally:
        _busy['b'] = False

w_gen_btn.on_click(lambda b: _generate(force=True))

# Auto-regenerate observers (cheap thanks to busy guard + auto-mode flag)
for w in [w_nx, w_ny, w_nz, w_xlen, w_ylen, w_zlen, w_depth, w_dip,
          w_seed, model_selector]:
    w.observe(_generate, names='value')
for _, wd, pw, _ in _PANELS.values():
    for w in wd.values():
        w.observe(_generate, names='value')
    if pw is not None:
        pw.observe(_generate, names='value')

_render_panel()
display(widgets.VBox([
    widgets.HTML('<h2>ResMill Interactive Explorer</h2>'),
    widgets.HTML('<i>Drag sliders, then click <b>Generate</b>. '
                  'Or tick <b>Auto-regenerate</b>.</i>'),
    grid_box,
    model_selector,
    param_area,
    widgets.HBox([w_gen_btn, w_auto]),
    status_label,
    output_area,
]))